# 05 — Master Dataset Build

This notebook comes after:

`04_Volatility_Features.ipynb`

Purpose:

Build the master analytical dataset by merging:

- standardized comparable country-year variables from Step 00;
- WGI governance dimensions and derived governance composite;
- GDP recovery outcomes;
- pre-shock volatility / stability features in the country-shock snapshot only.

This notebook creates:

- a master country-year panel without shock-window volatility leakage;
- a master country-shock structural snapshot panel with shock-specific volatility features;
- merge diagnostics;
- coverage diagnostics;
- a data dictionary;
- an audit workbook.

Important:

This notebook does **not**:

- choose the final analytical country sample;
- impute missing values;
- clip values;
- direction-align all variables for POSet;
- select final POSet ordering variables;
- run POSet analysis.

Recovery outcomes are merged only for later validation / interpretation.  
They must **not** be used as POSet ordering variables.

In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import shutil
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

COMPARABLE_LONG_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "00_Comparable_Raw_Files"
    / "Combined_Long"
    / "all_comparable_sources_long.csv"
)

WGI_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "WGI_Governance"
    / "wgi_final_project_ready_country_year_2000_2024.csv"
)

RECOVERY_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "Recovery"
    / "country_recovery_summary_dynamic_baseline_2007_2019.csv"
)

VOLATILITY_COUNTRY_WIDE_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "Volatility_Features"
    / "volatility_features_country_wide.csv"
)

VOLATILITY_COUNTRY_SHOCK_WIDE_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "Volatility_Features"
    / "volatility_features_country_shock_wide.csv"
)

GDP_STABILITY_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "Volatility_Features"
    / "gdp_growth_stability_features_country_shock.csv"
)

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "05_Master_Dataset_Build"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

PROJECT_START_YEAR = 2000
PROJECT_END_YEAR = 2024

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Comparable long file:", COMPARABLE_LONG_FILE.resolve())
print("WGI file:", WGI_FILE.resolve())
print("Recovery file:", RECOVERY_FILE.resolve())
print("Volatility country-wide file:", VOLATILITY_COUNTRY_WIDE_FILE.resolve())
print("Volatility country-shock-wide file:", VOLATILITY_COUNTRY_SHOCK_WIDE_FILE.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())

Run ID: 20260622_045424
Comparable long file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long.csv
WGI file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\WGI_Governance\wgi_final_project_ready_country_year_2000_2024.csv
Recovery file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Recovery\country_recovery_summary_dynamic_baseline_2007_2019.csv
Volatility country-wide file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Volatility_Features\volatility_features_country_wide.csv
Volatility country-shock-wide file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Proje

In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

# Core variables from the standardized comparable long dataset.
# Raw public debt source variables are excluded from the master analytical panel
# because canonical public debt is the master variable.
MASTER_CORE_VARIABLES = [
    "rd_gdp",
    "inflation_cpi",
    "unemployment_rate",
    "tertiary_education_25_64",
    "productivity_gdp_per_hour",
    "gdp_growth",
    "energy_import_dependency_raw",
    "public_debt_gdp_canonical",
]

SOURCE_DIAGNOSTIC_ONLY_VARIABLES = [
    "public_debt_gdp_eurostat",
    "public_debt_gdp_worldbank",
]

SHOCK_CONFIGS = [
    {
        "shock_id": "2007",
        "analysis_year": 2007,
        "recovery_column": "Recovery_2007",
        "pre_shock_window": "2000-2007",
    },
    {
        "shock_id": "2019",
        "analysis_year": 2019,
        "recovery_column": "Recovery_2019",
        "pre_shock_window": "2012-2019",
    },
]

EXPECTED_WGI_COMPOSITE_COL = "wgi_mahalanobis_ideal_score_full_wgi"

print("Master core variables:")
display(pd.DataFrame({"variable": MASTER_CORE_VARIABLES}))

print("Source diagnostic-only variables excluded from master core panel:")
display(pd.DataFrame({"variable": SOURCE_DIAGNOSTIC_ONLY_VARIABLES}))

print("Shock configs:")
display(pd.DataFrame(SHOCK_CONFIGS))

Master core variables:


,variable
0,rd_gdp
1,inflation_cpi
2,unemployment_rate
3,tertiary_education_25_64
4,productivity_gdp_per_hour
5,gdp_growth
6,energy_import_dependency_raw
7,public_debt_gdp_canonical


Source diagnostic-only variables excluded from master core panel:


,variable
0,public_debt_gdp_eurostat
1,public_debt_gdp_worldbank


Shock configs:


,shock_id,analysis_year,recovery_column,pre_shock_window
0,2007,2007,Recovery_2007,2000-2007
1,2019,2019,Recovery_2019,2012-2019


In [3]:
# ------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------

def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def clean_country_year(df, country_col="Country", year_col=None):
    out = df.copy()

    if country_col in out.columns:
        out[country_col] = out[country_col].astype(str).str.strip().str.upper()

    if year_col is not None and year_col in out.columns:
        out[year_col] = pd.to_numeric(out[year_col], errors="coerce")
        out = out.dropna(subset=[year_col]).copy()
        out[year_col] = out[year_col].astype(int)

    return out


def coalesce_columns(df, cols):
    existing = [col for col in cols if col in df.columns]

    if not existing:
        return pd.Series(np.nan, index=df.index)

    result = df[existing[0]].copy()

    for col in existing[1:]:
        result = result.combine_first(df[col])

    return result


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))

In [4]:
# ------------------------------------------------------
# LOAD COMPARABLE LONG DATA AND CREATE CORE WIDE PANEL
# ------------------------------------------------------

require_file(COMPARABLE_LONG_FILE, "Comparable long file")

comparable_long = pd.read_csv(COMPARABLE_LONG_FILE)

required_cols = {"Country", "Year", "variable", "Value"}
missing_cols = required_cols - set(comparable_long.columns)

if missing_cols:
    raise ValueError(f"Comparable long file missing required columns: {missing_cols}")

comparable_long = clean_country_year(comparable_long, "Country", "Year")
comparable_long["Value"] = pd.to_numeric(comparable_long["Value"], errors="coerce")
comparable_long = comparable_long.dropna(subset=["Country", "Year", "variable", "Value"]).copy()

if "country_name" not in comparable_long.columns:
    comparable_long["country_name"] = comparable_long["Country"]

available_variables = sorted(comparable_long["variable"].unique())
missing_master_core_variables = sorted(set(MASTER_CORE_VARIABLES) - set(available_variables))

if missing_master_core_variables:
    raise ValueError(f"Missing expected master core variables: {missing_master_core_variables}")

core_long = comparable_long[
    comparable_long["variable"].isin(MASTER_CORE_VARIABLES)
].copy()

source_diagnostic_long = comparable_long[
    comparable_long["variable"].isin(SOURCE_DIAGNOSTIC_ONLY_VARIABLES)
].copy()

# Duplicate check in core variables.
core_duplicate_check = (
    core_long
    .groupby(["Country", "Year", "variable"])
    .agg(
        row_count=("Value", "size"),
        unique_values=("Value", "nunique"),
        min_value=("Value", "min"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .query("row_count > 1")
)

core_duplicate_check.to_csv(
    DIAGNOSTICS_DIR / "master_core_duplicate_country_year_variable_check.csv",
    index=False,
)

if len(core_duplicate_check.query("unique_values > 1")) > 0:
    display(core_duplicate_check)
    raise ValueError("Conflicting duplicates found in master core variables.")

country_names_from_core = (
    core_long
    .groupby("Country")
    .agg(country_name_core=("country_name", "first"))
    .reset_index()
)

core_wide = (
    core_long
    .pivot_table(
        index=["Country", "Year"],
        columns="variable",
        values="Value",
        aggfunc="first",
    )
    .reset_index()
)

core_wide = core_wide.merge(country_names_from_core, on="Country", how="left")

front_cols = ["Country", "country_name_core", "Year"]
core_wide = core_wide[front_cols + MASTER_CORE_VARIABLES].copy()

core_wide = core_wide[
    (core_wide["Year"] >= PROJECT_START_YEAR)
    & (core_wide["Year"] <= PROJECT_END_YEAR)
].copy()

core_wide = core_wide.sort_values(["Country", "Year"]).reset_index(drop=True)

core_wide.to_csv(
    DIAGNOSTICS_DIR / "master_core_wide_snapshot.csv",
    index=False,
)

print("Comparable long rows:", len(comparable_long))
print("Core long rows:", len(core_long))
print("Core wide rows:", len(core_wide))
print("Core wide countries:", core_wide["Country"].nunique())
print("Core wide years:", core_wide["Year"].min(), "-", core_wide["Year"].max())

display(core_wide.head())

Comparable long rows: 13446
Core long rows: 11613
Core wide rows: 3584
Core wide countries: 169
Core wide years: 2000 - 2024


,Country,country_name_core,Year,rd_gdp,inflation_cpi,unemployment_rate,tertiary_education_25_64,productivity_gdp_per_hour,gdp_growth,energy_import_dependency_raw,public_debt_gdp_canonical
0,AGO,Angola,2000,NaN,NaN,NaN,NaN,NaN,NaN,-546.6450,NaN
1,AGO,Angola,2001,NaN,NaN,NaN,NaN,NaN,NaN,-487.0143,NaN
2,AGO,Angola,2002,NaN,NaN,NaN,NaN,NaN,NaN,-575.4543,NaN
3,AGO,Angola,2003,NaN,NaN,NaN,NaN,NaN,NaN,-520.1810,NaN
4,AGO,Angola,2004,NaN,NaN,NaN,NaN,NaN,NaN,-590.3533,NaN


In [5]:
# ------------------------------------------------------
# LOAD WGI GOVERNANCE DATA
# ------------------------------------------------------

require_file(WGI_FILE, "WGI project-ready file")

wgi = pd.read_csv(WGI_FILE)
wgi = clean_country_year(wgi, "Country", "Year")

if "country_name" in wgi.columns:
    wgi = wgi.rename(columns={"country_name": "country_name_wgi"})
else:
    wgi["country_name_wgi"] = np.nan

if EXPECTED_WGI_COMPOSITE_COL not in wgi.columns:
    raise ValueError(f"Expected WGI composite column missing: {EXPECTED_WGI_COMPOSITE_COL}")

wgi = wgi[
    (wgi["Year"] >= PROJECT_START_YEAR)
    & (wgi["Year"] <= PROJECT_END_YEAR)
].copy()

wgi_duplicate_check = (
    wgi
    .groupby(["Country", "Year"])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

wgi_duplicate_check.to_csv(
    DIAGNOSTICS_DIR / "wgi_duplicate_country_year_check.csv",
    index=False,
)

if not wgi_duplicate_check.empty:
    display(wgi_duplicate_check)
    raise ValueError("Duplicate WGI Country-Year rows found.")

wgi_merge_cols = [
    col for col in wgi.columns
    if col in ["Country", "country_name_wgi", "Year"]
    or col.startswith("wgi_")
]

wgi = wgi[wgi_merge_cols].copy()

wgi.to_csv(
    DIAGNOSTICS_DIR / "wgi_merge_input_snapshot.csv",
    index=False,
)

print("WGI rows:", len(wgi))
print("WGI countries:", wgi["Country"].nunique())
print("WGI years:", wgi["Year"].min(), "-", wgi["Year"].max())

display(wgi.head())

WGI rows: 5078
WGI countries: 216
WGI years: 2000 - 2024


,Country,country_name_wgi,Year,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score,wgi_va_estimate,wgi_pv_estimate,wgi_ge_estimate,wgi_rq_estimate,wgi_rl_estimate,wgi_cc_estimate,wgi_complete_six_selected_dimensions,wgi_composite_input_type,wgi_euclidean_distance_to_ideal_selected_space,wgi_euclidean_ideal_score,wgi_mahalanobis_distance_to_ideal_full_wgi,wgi_mahalanobis_ideal_score_full_wgi
0,ABW,Aruba,2004,64.6072,76.6424,68.7456,66.5859,67.4120,67.4139,0.4432,0.6292,0.8951,0.7433,0.6070,0.9663,True,score,77.5590,68.3367,2.4697,67.1844
1,ABW,Aruba,2005,75.2981,89.1503,75.6468,72.4106,70.5375,74.7178,1.0541,1.3615,1.2501,1.1126,0.7886,1.3271,True,score,59.9115,75.5412,2.6330,65.0153
2,ABW,Aruba,2006,75.4853,89.2011,75.5895,72.3508,70.4922,74.8834,1.0648,1.3644,1.2471,1.1088,0.7860,1.3353,True,score,59.8288,75.5750,2.6533,64.7456
3,ABW,Aruba,2007,75.4311,89.2851,75.6761,72.4548,70.5815,74.7648,1.0617,1.3694,1.2516,1.1154,0.7912,1.3294,True,score,59.7586,75.6036,2.6362,64.9726
4,ABW,Aruba,2008,75.3868,89.2748,75.7437,72.4087,70.4990,74.7963,1.0591,1.3687,1.2551,1.1125,0.7864,1.3310,True,score,59.7999,75.5868,2.6553,64.7188


In [6]:
# ------------------------------------------------------
# LOAD RECOVERY OUTCOMES
# ------------------------------------------------------

require_file(RECOVERY_FILE, "Recovery file")

recovery = pd.read_csv(RECOVERY_FILE)
recovery = clean_country_year(recovery, "Country", None)

if "country_name" in recovery.columns:
    recovery = recovery.rename(columns={"country_name": "country_name_recovery"})
else:
    recovery["country_name_recovery"] = np.nan

expected_recovery_cols = ["Recovery_2007", "Recovery_2019"]
missing_recovery_cols = sorted(set(expected_recovery_cols) - set(recovery.columns))

if missing_recovery_cols:
    raise ValueError(f"Recovery file missing expected columns: {missing_recovery_cols}")

recovery_duplicate_check = (
    recovery
    .groupby("Country")
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

recovery_duplicate_check.to_csv(
    DIAGNOSTICS_DIR / "recovery_duplicate_country_check.csv",
    index=False,
)

if not recovery_duplicate_check.empty:
    display(recovery_duplicate_check)
    raise ValueError("Duplicate recovery Country rows found.")

# Keep main recovery columns plus audit fields.
recovery_cols = [
    col for col in recovery.columns
    if col == "Country"
    or col == "country_name_recovery"
    or col.startswith("Recovery_")
]

recovery = recovery[recovery_cols].copy()

recovery.to_csv(
    DIAGNOSTICS_DIR / "recovery_merge_input_snapshot.csv",
    index=False,
)

print("Recovery rows:", len(recovery))
print("Recovery countries:", recovery["Country"].nunique())

display(recovery.head())

Recovery rows: 44
Recovery countries: 44


,Country,country_name_recovery,Recovery_2007,Recovery_2019,Recovery_2007_recovery_status,Recovery_2007_shock_detection_status,Recovery_2007_first_negative_growth_year,Recovery_2007_baseline_year,Recovery_2007_baseline_available_in_growth_file,Recovery_2007_recovery_year,Recovery_2007_years_from_baseline_to_recovery,Recovery_2007_years_after_first_negative_to_recovery,Recovery_2007_last_index_year,Recovery_2007_last_index_value,Recovery_2019_recovery_status,Recovery_2019_shock_detection_status,Recovery_2019_first_negative_growth_year,Recovery_2019_baseline_year,Recovery_2019_baseline_available_in_growth_file,Recovery_2019_recovery_year,Recovery_2019_years_from_baseline_to_recovery,Recovery_2019_years_after_first_negative_to_recovery,Recovery_2019_last_index_year,Recovery_2019_last_index_value
0,AUS,Australia,0.0000,0.0000,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,0.0000,0.0000,NaN,NaN,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,0.0000,0.0000,NaN,NaN
1,AUT,Austria,2.0000,2.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2011.0000,3.0000,2.0000,2011.0000,101.0314,recovered,negative_growth_found,2020.0000,2019.0000,True,2022.0000,3.0000,2.0000,2022.0000,103.5338
2,BEL,Belgium,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,100.7522,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,101.1617
3,BRA,Brazil,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,107.3929,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,101.3298
4,CAN,Canada,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,100.0856,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,100.6125


In [7]:
# ------------------------------------------------------
# LOAD VOLATILITY FEATURES
# ------------------------------------------------------

if VOLATILITY_COUNTRY_WIDE_FILE.exists():
    volatility_country_wide = pd.read_csv(VOLATILITY_COUNTRY_WIDE_FILE)
    volatility_country_wide = clean_country_year(volatility_country_wide, "Country", None)

    if "country_name" in volatility_country_wide.columns:
        volatility_country_wide = volatility_country_wide.rename(columns={"country_name": "country_name_volatility"})
    else:
        volatility_country_wide["country_name_volatility"] = np.nan

    volatility_country_duplicate_check = (
        volatility_country_wide
        .groupby("Country")
        .size()
        .reset_index(name="row_count")
        .query("row_count > 1")
    )

    volatility_country_duplicate_check.to_csv(
        DIAGNOSTICS_DIR / "volatility_country_duplicate_check.csv",
        index=False,
    )

    if not volatility_country_duplicate_check.empty:
        display(volatility_country_duplicate_check)
        raise ValueError("Duplicate volatility country rows found.")
else:
    volatility_country_wide = pd.DataFrame(columns=["Country", "country_name_volatility"])
    print("WARNING: Volatility country-wide file not found. Country-wide volatility merge will be empty.")

if VOLATILITY_COUNTRY_SHOCK_WIDE_FILE.exists():
    volatility_country_shock_wide = pd.read_csv(VOLATILITY_COUNTRY_SHOCK_WIDE_FILE)
    volatility_country_shock_wide = clean_country_year(volatility_country_shock_wide, "Country", None)

    # IMPORTANT:
    # shock_id may be read as int64 from CSV even though the master snapshot uses strings.
    # Force string type here to avoid object/int merge errors later.
    if "shock_id" in volatility_country_shock_wide.columns:
        volatility_country_shock_wide["shock_id"] = (
            volatility_country_shock_wide["shock_id"]
            .astype(str)
            .str.strip()
        )

    if "country_name" in volatility_country_shock_wide.columns:
        volatility_country_shock_wide = volatility_country_shock_wide.rename(columns={"country_name": "country_name_volatility_shock"})
    else:
        volatility_country_shock_wide["country_name_volatility_shock"] = np.nan

    volatility_country_shock_duplicate_check = (
        volatility_country_shock_wide
        .groupby(["Country", "shock_id"])
        .size()
        .reset_index(name="row_count")
        .query("row_count > 1")
    )

    volatility_country_shock_duplicate_check.to_csv(
        DIAGNOSTICS_DIR / "volatility_country_shock_duplicate_check.csv",
        index=False,
    )

    if not volatility_country_shock_duplicate_check.empty:
        display(volatility_country_shock_duplicate_check)
        raise ValueError("Duplicate volatility country-shock rows found.")
else:
    volatility_country_shock_wide = pd.DataFrame(columns=["Country", "shock_id"])
    print("WARNING: Volatility country-shock-wide file not found. Shock-wide volatility merge will be empty.")

if GDP_STABILITY_FILE.exists():
    gdp_stability = pd.read_csv(GDP_STABILITY_FILE)
    gdp_stability = clean_country_year(gdp_stability, "Country", None)

    if "shock_id" in gdp_stability.columns:
        gdp_stability["shock_id"] = (
            gdp_stability["shock_id"]
            .astype(str)
            .str.strip()
        )
else:
    gdp_stability = pd.DataFrame()
    print("WARNING: GDP stability file not found. Preferred stability diagnostics unavailable.")

print("Volatility country-wide rows:", len(volatility_country_wide))
print("Volatility country-shock-wide rows:", len(volatility_country_shock_wide))
print("GDP stability rows:", len(gdp_stability))

display(volatility_country_wide.head())
display(volatility_country_shock_wide.head())

Volatility country-wide rows: 54
Volatility country-shock-wide rows: 106
GDP stability rows: 88


,Country,country_name_volatility,gdp_growth_volatility_std_pre_2007,gdp_growth_volatility_std_pre_2019,inflation_volatility_std_pre_2007,inflation_volatility_std_pre_2019,productivity_level_volatility_std_pre_2007,productivity_level_volatility_std_pre_2019,unemployment_volatility_std_pre_2007,unemployment_volatility_std_pre_2019,gdp_growth_volatility_mad_pre_2007,gdp_growth_volatility_mad_pre_2019,inflation_volatility_mad_pre_2007,inflation_volatility_mad_pre_2019,productivity_level_volatility_mad_pre_2007,productivity_level_volatility_mad_pre_2019,unemployment_volatility_mad_pre_2007,unemployment_volatility_mad_pre_2019,gdp_growth_stability_negative_std_pre_2007,gdp_growth_stability_negative_std_pre_2019,inflation_stability_negative_std_pre_2007,inflation_stability_negative_std_pre_2019,productivity_level_stability_negative_std_pre_2007,productivity_level_stability_negative_std_pre_2019,unemployment_stability_negative_std_pre_2007,unemployment_stability_negative_std_pre_2019,gdp_growth_stability_negative_mad_pre_2007,gdp_growth_stability_negative_mad_pre_2019,inflation_stability_negative_mad_pre_2007,inflation_stability_negative_mad_pre_2019,productivity_level_stability_negative_mad_pre_2007,productivity_level_stability_negative_mad_pre_2019,unemployment_stability_negative_mad_pre_2007,unemployment_stability_negative_mad_pre_2019,gdp_growth_stability_std_0_100_pre_2007,gdp_growth_stability_std_0_100_pre_2019,inflation_stability_std_0_100_pre_2007,inflation_stability_std_0_100_pre_2019,productivity_level_stability_std_0_100_pre_2007,productivity_level_stability_std_0_100_pre_2019,unemployment_stability_std_0_100_pre_2007,unemployment_stability_std_0_100_pre_2019,gdp_growth_stability_mad_0_100_pre_2007,gdp_growth_stability_mad_0_100_pre_2019,inflation_stability_mad_0_100_pre_2007,inflation_stability_mad_0_100_pre_2019,productivity_level_stability_mad_0_100_pre_2007,productivity_level_stability_mad_0_100_pre_2019,unemployment_stability_mad_0_100_pre_2007,unemployment_stability_mad_0_100_pre_2019,gdp_growth_mean_value_pre_2007,gdp_growth_mean_value_pre_2019,inflation_mean_value_pre_2007,inflation_mean_value_pre_2019,productivity_level_mean_value_pre_2007,productivity_level_mean_value_pre_2019,unemployment_mean_value_pre_2007,unemployment_mean_value_pre_2019,gdp_growth_median_value_pre_2007,gdp_growth_median_value_pre_2019,inflation_median_value_pre_2007,inflation_median_value_pre_2019,productivity_level_median_value_pre_2007,productivity_level_median_value_pre_2019,unemployment_median_value_pre_2007,unemployment_median_value_pre_2019,gdp_growth_change_first_to_last_pre_2007,gdp_growth_change_first_to_last_pre_2019,inflation_change_first_to_last_pre_2007,inflation_change_first_to_last_pre_2019,productivity_level_change_first_to_last_pre_2007,productivity_level_change_first_to_last_pre_2019,unemployment_change_first_to_last_pre_2007,unemployment_change_first_to_last_pre_2019,gdp_growth_years_available_pre_2007,gdp_growth_years_available_pre_2019,inflation_years_available_pre_2007,inflation_years_available_pre_2019,productivity_level_years_available_pre_2007,productivity_level_years_available_pre_2019,unemployment_years_available_pre_2007,unemployment_years_available_pre_2019,gdp_growth_coverage_rate_in_window_pre_2007,gdp_growth_coverage_rate_in_window_pre_2019,inflation_coverage_rate_in_window_pre_2007,inflation_coverage_rate_in_window_pre_2019,productivity_level_coverage_rate_in_window_pre_2007,productivity_level_coverage_rate_in_window_pre_2019,unemployment_coverage_rate_in_window_pre_2007,unemployment_coverage_rate_in_window_pre_2019
0,ARG,Argentina,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.9555,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.8142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.9555,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.8142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.4398,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.6889,NaN,NaN,NaN,43.9128,NaN,NaN,NaN,8.2018,NaN,NaN,NaN,43.9128,NaN,NaN,NaN,8.2305,NaN,NaN,NaN,19.2711,NaN,NaN,NaN,1.3390,NaN,NaN,NaN,2.0000,NaN,NaN,NaN,4.0000,NaN,NaN,NaN,0.25

,Country,country_name_volatility_shock,shock_id,shock_label,window_start,window_end,gdp_growth_volatility_std_pre_2007,gdp_growth_volatility_std_pre_2019,inflation_volatility_std_pre_2007,inflation_volatility_std_pre_2019,productivity_level_volatility_std_pre_2007,productivity_level_volatility_std_pre_2019,unemployment_volatility_std_pre_2007,unemployment_volatility_std_pre_2019,gdp_growth_volatility_mad_pre_2007,gdp_growth_volatility_mad_pre_2019,inflation_volatility_mad_pre_2007,inflation_volatility_mad_pre_2019,productivity_level_volatility_mad_pre_2007,productivity_level_volatility_mad_pre_2019,unemployment_volatility_mad_pre_2007,unemployment_volatility_mad_pre_2019,gdp_growth_stability_negative_std_pre_2007,gdp_growth_stability_negative_std_pre_2019,inflation_stability_negative_std_pre_2007,inflation_stability_negative_std_pre_2019,productivity_level_stability_negative_std_pre_2007,productivity_level_stability_negative_std_pre_2019,unemployment_stability_negative_std_pre_2007,unemployment_stability_negative_std_pre_2019,gdp_growth_stability_negative_mad_pre_2007,gdp_growth_stability_negative_mad_pre_2019,inflation_stability_negative_mad_pre_2007,inflation_stability_negative_mad_pre_2019,productivity_level_stability_negative_mad_pre_2007,productivity_level_stability_negative_mad_pre_2019,unemployment_stability_negative_mad_pre_2007,unemployment_stability_negative_mad_pre_2019,gdp_growth_stability_std_0_100_pre_2007,gdp_growth_stability_std_0_100_pre_2019,inflation_stability_std_0_100_pre_2007,inflation_stability_std_0_100_pre_2019,productivity_level_stability_std_0_100_pre_2007,productivity_level_stability_std_0_100_pre_2019,unemployment_stability_std_0_100_pre_2007,unemployment_stability_std_0_100_pre_2019,gdp_growth_stability_mad_0_100_pre_2007,gdp_growth_stability_mad_0_100_pre_2019,inflation_stability_mad_0_100_pre_2007,inflation_stability_mad_0_100_pre_2019,productivity_level_stability_mad_0_100_pre_2007,productivity_level_stability_mad_0_100_pre_2019,unemployment_stability_mad_0_100_pre_2007,unemployment_stability_mad_0_100_pre_2019,gdp_growth_mean_value_pre_2007,gdp_growth_mean_value_pre_2019,inflation_mean_value_pre_2007,inflation_mean_value_pre_2019,productivity_level_mean_value_pre_2007,productivity_level_mean_value_pre_2019,unemployment_mean_value_pre_2007,unemployment_mean_value_pre_2019,gdp_growth_median_value_pre_2007,gdp_growth_median_value_pre_2019,inflation_median_value_pre_2007,inflation_median_value_pre_2019,productivity_level_median_value_pre_2007,productivity_level_median_value_pre_2019,unemployment_median_value_pre_2007,unemployment_median_value_pre_2019,gdp_growth_change_first_to_last_pre_2007,gdp_growth_change_first_to_last_pre_2019,inflation_change_first_to_last_pre_2007,inflation_change_first_to_last_pre_2019,productivity_level_change_first_to_last_pre_2007,productivity_level_change_first_to_last_pre_2019,unemployment_change_first_to_last_pre_2007,unemployment_change_first_to_last_pre_2019,gdp_growth_years_available_pre_2007,gdp_growth_years_available_pre_2019,inflation_years_available_pre_2007,inflation_years_available_pre_2019,productivity_level_years_available_pre_2007,productivity_level_years_available_pre_2019,unemployment_years_available_pre_2007,unemployment_years_available_pre_2019,gdp_growth_coverage_rate_in_window_pre_2007,gdp_growth_coverage_rate_in_window_pre_2019,inflation_coverage_rate_in_window_pre_2007,inflation_coverage_rate_in_window_pre_2019,productivity_level_coverage_rate_in_window_pre_2007,productivity_level_coverage_rate_in_window_pre_2019,unemployment_coverage_rate_in_window_pre_2007,unemployment_coverage_rate_in_window_pre_2019
0,AUS,Australia,2007,shock_2007,2000,2007,0.7130,NaN,0.8605,NaN,1.8835,NaN,0.8455,NaN,0.5666,NaN,0.7146,NaN,1.5233,NaN,0.7175,NaN,-0.7130,NaN,-0.8605,NaN,-1.8835,NaN,-0.8455,NaN,-0.5666,NaN,-0.7146,NaN,-1.5233,NaN,-0.7175,NaN,100.0000,NaN,97.0362,NaN,69.5619,NaN,86.6022,NaN,100.0000,NaN,97.1921,NaN,70.7432,NaN,85.2081,NaN,3.3289,NaN,3.1871,NaN,57.4

In [8]:
# ------------------------------------------------------
# MERGE COUNTRY-YEAR MASTER PANEL
# ------------------------------------------------------
# Merge order:
# 1. core country-year variables
# 2. WGI country-year variables
# 3. country-level recovery outcomes
#
# IMPORTANT:
# Pre-shock volatility features are NOT merged into the country-year panel.
# They are shock-window features, so merging both 2007 and 2019 volatility
# features onto every country-year row can create confusing post-shock leakage
# in the later structural snapshot.
#
# Volatility features are merged only in the country-shock snapshot below.

master_country_year_panel = core_wide.merge(
    wgi,
    on=["Country", "Year"],
    how="left",
    validate="many_to_one",
)

master_country_year_panel = master_country_year_panel.merge(
    recovery,
    on="Country",
    how="left",
    validate="many_to_one",
)

master_country_year_panel["country_name"] = coalesce_columns(
    master_country_year_panel,
    [
        "country_name_core",
        "country_name_wgi",
        "country_name_recovery",
        "Country",
    ],
)

# Put identifiers first.
identifier_cols = ["Country", "country_name", "Year"]

metadata_name_cols = [
    col for col in [
        "country_name_core",
        "country_name_wgi",
        "country_name_recovery",
    ]
    if col in master_country_year_panel.columns
]

remaining_cols = [
    col for col in master_country_year_panel.columns
    if col not in identifier_cols + metadata_name_cols
]

master_country_year_panel = master_country_year_panel[
    identifier_cols + remaining_cols + metadata_name_cols
].sort_values(["Country", "Year"]).reset_index(drop=True)

master_country_year_panel.to_csv(
    PROCESSED_DIR / "master_country_year_panel.csv",
    index=False,
)

master_country_year_panel.to_csv(
    PROCESSED_DIR / "master_country_year_panel_2000_2024.csv",
    index=False,
)

print("Master country-year panel rows:", len(master_country_year_panel))
print("Countries:", master_country_year_panel["Country"].nunique())
print("Years:", master_country_year_panel["Year"].min(), "-", master_country_year_panel["Year"].max())
print("Columns:", len(master_country_year_panel.columns))
print("Volatility features merged into country-year panel: NO")

display(master_country_year_panel.head())

Master country-year panel rows: 3584
Countries: 169
Years: 2000 - 2024
Columns: 54
Volatility features merged into country-year panel: NO


,Country,country_name,Year,rd_gdp,inflation_cpi,unemployment_rate,tertiary_education_25_64,productivity_gdp_per_hour,gdp_growth,energy_import_dependency_raw,public_debt_gdp_canonical,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score,wgi_va_estimate,wgi_pv_estimate,wgi_ge_estimate,wgi_rq_estimate,wgi_rl_estimate,wgi_cc_estimate,wgi_complete_six_selected_dimensions,wgi_composite_input_type,wgi_euclidean_distance_to_ideal_selected_space,wgi_euclidean_ideal_score,wgi_mahalanobis_distance_to_ideal_full_wgi,wgi_mahalanobis_ideal_score_full_wgi,Recovery_2007,Recovery_2019,Recovery_2007_recovery_status,Recovery_2007_shock_detection_status,Recovery_2007_first_negative_growth_year,Recovery_2007_baseline_year,Recovery_2007_baseline_available_in_growth_file,Recovery_2007_recovery_year,Recovery_2007_years_from_baseline_to_recovery,Recovery_2007_years_after_first_negative_to_recovery,Recovery_2007_last_index_year,Recovery_2007_last_index_value,Recovery_2019_recovery_status,Recovery_2019_shock_detection_status,Recovery_2019_first_negative_growth_year,Recovery_2019_baseline_year,Recovery_2019_baseline_available_in_growth_file,Recovery_2019_recovery_year,Recovery_2019_years_from_baseline_to_recovery,Recovery_2019_years_after_first_negative_to_recovery,Recovery_2019_last_index_year,Recovery_2019_last_index_value,country_name_core,country_name_wgi,country_name_recovery
0,AGO,Angola,2000,NaN,NaN,NaN,NaN,NaN,NaN,-546.6450,NaN,27.4713,24.1205,29.6775,26.3701,31.7802,21.9370,-1.6785,-2.4456,-1.1141,-1.8066,-1.4633,-1.2804,True,score,179.2554,26.8193,5.8656,22.0635,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Angola,Angola,NaN
1,AGO,Angola,2001,NaN,NaN,NaN,NaN,NaN,NaN,-487.0143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Angola,NaN,NaN
2,AGO,Angola,2002,NaN,NaN,NaN,NaN,NaN,NaN,-575.4543,NaN,32.6144,34.8778,27.4225,29.5268,31.2147,21.4569,-1.3847,-1.8158,-1.2301,-1.6064,-1.4962,-1.3041,True,score,172.9628,29.3882,4.8841,35.1052,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Angola,Angola,NaN
3,AGO,Angola,2003,NaN,NaN,NaN,NaN,NaN,NaN,-520.1810,NaN,33.9254,44.5306,22.9871,28.9379,28.1348,21.0722,-1.3097,-1.2507,-1.4582,-1.6438,-1.6751,-1.3231,True,score,172.6781,29.5045,4.5973,38.9157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Angola,Angola,NaN
4,AGO,Angola,2004,NaN,NaN,NaN,NaN,NaN,NaN,-590.3533,NaN,33.8036,46.0945,21.5584,32.3238,31.9420,23.1021,-1.3167,-1.1591,-1.5317,-1.4291,-1.4539,-1.2229,True,score,169.0151,30.9999,4.3365,42.3803,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Angola,Angola,NaN


In [9]:
# ------------------------------------------------------
# MERGE DIAGNOSTICS
# ------------------------------------------------------

core_keys = core_wide[["Country", "Year"]].drop_duplicates()
wgi_keys = wgi[["Country", "Year"]].drop_duplicates()

merge_key_diagnostics = pd.DataFrame([
    {
        "source": "core_country_year_panel",
        "rows": len(core_wide),
        "unique_countries": core_wide["Country"].nunique(),
        "unique_country_years": len(core_keys),
        "min_year": core_wide["Year"].min(),
        "max_year": core_wide["Year"].max(),
    },
    {
        "source": "wgi_country_year",
        "rows": len(wgi),
        "unique_countries": wgi["Country"].nunique(),
        "unique_country_years": len(wgi_keys),
        "min_year": wgi["Year"].min(),
        "max_year": wgi["Year"].max(),
    },
    {
        "source": "recovery_country_level",
        "rows": len(recovery),
        "unique_countries": recovery["Country"].nunique(),
        "unique_country_years": np.nan,
        "min_year": np.nan,
        "max_year": np.nan,
    },
    {
        "source": "volatility_country_level",
        "rows": len(volatility_country_wide),
        "unique_countries": volatility_country_wide["Country"].nunique() if not volatility_country_wide.empty else 0,
        "unique_country_years": np.nan,
        "min_year": np.nan,
        "max_year": np.nan,
    },
])

merge_key_diagnostics.to_csv(
    DIAGNOSTICS_DIR / "master_merge_key_diagnostics.csv",
    index=False,
)

core_country_set = set(core_wide["Country"].unique())
wgi_country_set = set(wgi["Country"].unique())
recovery_country_set = set(recovery["Country"].unique())
volatility_country_set = set(volatility_country_wide["Country"].unique()) if not volatility_country_wide.empty else set()

country_overlap_diagnostics = pd.DataFrame([
    {
        "comparison": "core_countries",
        "count": len(core_country_set),
    },
    {
        "comparison": "wgi_countries",
        "count": len(wgi_country_set),
    },
    {
        "comparison": "recovery_countries",
        "count": len(recovery_country_set),
    },
    {
        "comparison": "volatility_countries",
        "count": len(volatility_country_set),
    },
    {
        "comparison": "core_with_wgi",
        "count": len(core_country_set & wgi_country_set),
    },
    {
        "comparison": "core_with_recovery",
        "count": len(core_country_set & recovery_country_set),
    },
    {
        "comparison": "core_with_volatility",
        "count": len(core_country_set & volatility_country_set),
    },
    {
        "comparison": "core_with_wgi_recovery_volatility",
        "count": len(core_country_set & wgi_country_set & recovery_country_set & volatility_country_set),
    },
])

country_overlap_diagnostics.to_csv(
    DIAGNOSTICS_DIR / "master_country_overlap_diagnostics.csv",
    index=False,
)

wgi_only_countries = sorted(wgi_country_set - core_country_set)
recovery_not_in_core_countries = sorted(recovery_country_set - core_country_set)
core_without_wgi_countries = sorted(core_country_set - wgi_country_set)
core_without_recovery_countries = sorted(core_country_set - recovery_country_set)

country_merge_gaps = pd.DataFrame(
    [{"gap_type": "wgi_only_not_in_core", "Country": c} for c in wgi_only_countries]
    + [{"gap_type": "recovery_not_in_core", "Country": c} for c in recovery_not_in_core_countries]
    + [{"gap_type": "core_without_wgi", "Country": c} for c in core_without_wgi_countries]
    + [{"gap_type": "core_without_recovery", "Country": c} for c in core_without_recovery_countries]
)

country_merge_gaps.to_csv(
    DIAGNOSTICS_DIR / "master_country_merge_gaps.csv",
    index=False,
)

display(merge_key_diagnostics)
display(country_overlap_diagnostics)
display(country_merge_gaps.head(100))

,source,rows,unique_countries,unique_country_years,min_year,max_year
0,core_country_year_panel,3584,169,3584.0000,2000.0000,2024.0000
1,wgi_country_year,5078,216,5078.0000,2000.0000,2024.0000
2,recovery_country_level,44,44,NaN,NaN,NaN
3,volatility_country_level,54,54,NaN,NaN,NaN


,comparison,count
0,core_countries,169
1,wgi_countries,216
2,recovery_countries,44
3,volatility_countries,54
4,core_with_wgi,168
5,core_with_recovery,44
6,core_with_volatility,54
7,core_with_wgi_recovery_volatility,44


,gap_type,Country
0,wgi_only_not_in_core,ABW
1,wgi_only_not_in_core,AFG
2,wgi_only_not_in_core,AIA
3,wgi_only_not_in_core,ANT
4,wgi_only_not_in_core,ASM
5,wgi_only_not_in_core,ATG
6,wgi_only_not_in_core,BDI
7,wgi_only_not_in_core,BMU
8,wgi_only_not_in_core,CAF
9,wgi_only_not_in_core,COK


In [10]:
# ------------------------------------------------------
# MASTER VARIABLE COVERAGE DIAGNOSTICS
# ------------------------------------------------------
# Country-year panel coverage only.
# Shock-window volatility features are intentionally not included here because
# they belong to the country-shock snapshot.

wgi_score_cols = [
    col for col in master_country_year_panel.columns
    if re.match(r"^wgi_(va|pv|ge|rq|rl|cc)_score$", col)
]

wgi_composite_cols = [
    col for col in [
        "wgi_mahalanobis_ideal_score_full_wgi",
        "wgi_euclidean_ideal_score",
    ]
    if col in master_country_year_panel.columns
]

recovery_cols = [
    col for col in ["Recovery_2007", "Recovery_2019"]
    if col in master_country_year_panel.columns
]

# Keep this explicitly empty in the country-year panel.
# Volatility/stability coverage is checked in the structural snapshot coverage table.
volatility_cols = []

master_key_variable_cols = (
    MASTER_CORE_VARIABLES
    + wgi_score_cols
    + wgi_composite_cols
    + recovery_cols
)

master_key_variable_cols = [
    col for col in master_key_variable_cols
    if col in master_country_year_panel.columns
]

coverage_rows = []

for col in master_key_variable_cols:
    values = master_country_year_panel[col]

    coverage_rows.append({
        "column": col,
        "rows_non_missing": int(values.notna().sum()),
        "rows_total": len(master_country_year_panel),
        "row_coverage_rate": values.notna().mean(),
        "countries_with_any_data": int(master_country_year_panel.loc[values.notna(), "Country"].nunique()),
        "years_with_any_data": int(master_country_year_panel.loc[values.notna(), "Year"].nunique()),
        "min_value": pd.to_numeric(values, errors="coerce").min(),
        "median_value": pd.to_numeric(values, errors="coerce").median(),
        "max_value": pd.to_numeric(values, errors="coerce").max(),
        "role": (
            "core_structural_source"
            if col in MASTER_CORE_VARIABLES
            else "wgi_score"
            if col in wgi_score_cols
            else "wgi_composite"
            if col in wgi_composite_cols
            else "recovery_outcome_validation_only"
            if col in recovery_cols
            else "review"
        ),
    })

master_variable_coverage_summary = pd.DataFrame(coverage_rows).sort_values(
    ["role", "row_coverage_rate", "column"],
    ascending=[True, False, True],
).reset_index(drop=True)

master_variable_coverage_summary.to_csv(
    DIAGNOSTICS_DIR / "master_variable_coverage_summary.csv",
    index=False,
)

country_coverage_summary = master_country_year_panel.copy()
country_coverage_summary["core_variables_available"] = country_coverage_summary[MASTER_CORE_VARIABLES].notna().sum(axis=1)
country_coverage_summary["wgi_composite_available"] = (
    country_coverage_summary[EXPECTED_WGI_COMPOSITE_COL].notna()
    if EXPECTED_WGI_COMPOSITE_COL in country_coverage_summary.columns
    else False
)
country_coverage_summary["recovery_2007_available"] = (
    country_coverage_summary["Recovery_2007"].notna()
    if "Recovery_2007" in country_coverage_summary.columns
    else False
)
country_coverage_summary["recovery_2019_available"] = (
    country_coverage_summary["Recovery_2019"].notna()
    if "Recovery_2019" in country_coverage_summary.columns
    else False
)

master_country_coverage_summary = (
    country_coverage_summary
    .groupby(["Country", "country_name"])
    .agg(
        years_in_master=("Year", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        mean_core_variables_available=("core_variables_available", "mean"),
        min_core_variables_available=("core_variables_available", "min"),
        max_core_variables_available=("core_variables_available", "max"),
        years_with_wgi_composite=("wgi_composite_available", "sum"),
        has_recovery_2007=("recovery_2007_available", "max"),
        has_recovery_2019=("recovery_2019_available", "max"),
    )
    .reset_index()
)

master_country_coverage_summary["mean_core_coverage_rate"] = (
    master_country_coverage_summary["mean_core_variables_available"] / len(MASTER_CORE_VARIABLES)
)

master_country_coverage_summary.to_csv(
    DIAGNOSTICS_DIR / "master_country_coverage_summary.csv",
    index=False,
)

display(master_variable_coverage_summary)
display(master_country_coverage_summary.head(80))

,column,rows_non_missing,rows_total,row_coverage_rate,countries_with_any_data,years_with_any_data,min_value,median_value,max_value,role
0,energy_import_dependency_raw,3151,3584,0.8792,145,24,-3300.1473,24.3911,450.1815,core_structural_source
1,public_debt_gdp_canonical,1756,3584,0.4900,112,25,-1.1707,50.2023,209.4000,core_structural_source
2,unemployment_rate,1239,3584,0.3457,53,25,0.2510,6.6100,36.0250,core_structural_source
3,inflation_cpi,1170,3584,0.3265,48,25,-4.4475,2.5930,219.8845,core_structural_source
4,gdp_growth,1092,3584,0.3047,44,25,-16.0400,2.7012,24.6240,core_structural_source
5,rd_gdp,1076,3584,0.3002,47,25,0.1401,1.5417,6.9589,core_structural_source
6,productivity_gdp_per_hour,1003,3584,0.2799,41,25,11.7448,51.2618,142.9044,core_structural_source
7,tertiary_education_25_64,973,3584,0.2715,48,25,5.4441,30.7121,64.6622,core_structural_source
8,Recovery_2007,1100,3584,0.3069,44,25,0.0000,2.0000,20.0000,recovery_outcome_validation_only
9,Recovery_2019,1075,3584,0.2999,43,25,0.0000,1.0000,2.0000,recovery_outcome_validation_only


,Country,country_name,years_in_master,min_year,max_year,mean_core_variables_available,min_core_variables_available,max_core_variables_available,years_with_wgi_composite,has_recovery_2007,has_recovery_2019,mean_core_coverage_rate
0,AGO,Angola,23,2000,2022,1.0000,1,1,22,False,False,0.1250
1,ALB,Albania,25,2000,2024,1.5200,1,2,24,False,False,0.1900
2,AND,Andorra,7,2018,2024,1.0000,1,1,7,False,False,0.1250
3,ARE,United Arab Emirates,23,2000,2022,1.0435,1,2,22,False,False,0.1304
4,ARG,Argentina,25,2000,2024,3.2000,1,5,24,False,False,0.4000
5,ARM,Armenia,25,2000,2024,1.1600,1,2,24,False,False,0.1450
6,AUS,Australia,25,2000,2024,7.3600,4,8,24,True,True,0.9200
7,AUT,Austria,25,2000,2024,7.8000,7,8,24,True,True,0.9750
8,AZE,Azerbaijan,24,2000,2023,1.1250,1,2,23,False,False,0.1406
9,BEL,Belgium,25,2000,2024,7.9600,7,8,24,True,True,0.9950


In [11]:
# ------------------------------------------------------
# BUILD MASTER STRUCTURAL SNAPSHOT BY SHOCK
# ------------------------------------------------------
# One row per Country x Shock.
#
# For shock 2007:
#     structural variables are taken from Year = 2007
#     recovery outcome = Recovery_2007
#     volatility features = pre-2007 window features only
#
# For shock 2019:
#     structural variables are taken from Year = 2019
#     recovery outcome = Recovery_2019
#     volatility features = pre-2019 window features only
#
# This avoids carrying pre-2019 volatility into the 2007 snapshot or
# pre-2007 volatility into the 2019 snapshot.

shock_snapshot_rows = []

for config in SHOCK_CONFIGS:
    shock_id = str(config["shock_id"])
    analysis_year = config["analysis_year"]
    recovery_column = config["recovery_column"]
    pre_shock_window = config["pre_shock_window"]

    snapshot = master_country_year_panel[
        master_country_year_panel["Year"] == analysis_year
    ].copy()

    snapshot["shock_id"] = shock_id
    snapshot["analysis_year"] = analysis_year
    snapshot["pre_shock_window"] = pre_shock_window
    snapshot["recovery_outcome_column"] = recovery_column

    if recovery_column in snapshot.columns:
        snapshot["Recovery"] = snapshot[recovery_column]
    else:
        snapshot["Recovery"] = np.nan

    # Merge only the volatility columns for the current shock.
    if not volatility_country_shock_wide.empty:
        vol_shock = volatility_country_shock_wide.copy()
        vol_shock["shock_id"] = vol_shock["shock_id"].astype(str).str.strip()

        vol_shock = vol_shock[
            vol_shock["shock_id"] == shock_id
        ].copy()

        shock_suffix = f"_pre_{shock_id}"

        keep_cols = [
            col for col in vol_shock.columns
            if col in ["Country", "shock_id", "shock_label", "window_start", "window_end"]
            or col.endswith(shock_suffix)
        ]

        vol_shock_merge = vol_shock[keep_cols].copy()

        # Avoid duplicate structural shock metadata columns when present.
        for col in ["shock_label", "window_start", "window_end"]:
            if col in vol_shock_merge.columns and col in snapshot.columns:
                vol_shock_merge = vol_shock_merge.drop(columns=[col])

        snapshot["shock_id"] = snapshot["shock_id"].astype(str).str.strip()
        vol_shock_merge["shock_id"] = vol_shock_merge["shock_id"].astype(str).str.strip()

        snapshot = snapshot.merge(
            vol_shock_merge,
            on=["Country", "shock_id"],
            how="left",
            validate="many_to_one",
        )

    shock_snapshot_rows.append(snapshot)

master_structural_snapshot_by_shock = pd.concat(shock_snapshot_rows, ignore_index=True)

# Safety check: no cross-window volatility leakage.
leakage_rows = []

for shock_id in master_structural_snapshot_by_shock["shock_id"].astype(str).unique():
    other_shock_ids = [
        str(config["shock_id"])
        for config in SHOCK_CONFIGS
        if str(config["shock_id"]) != shock_id
    ]

    group = master_structural_snapshot_by_shock[
        master_structural_snapshot_by_shock["shock_id"].astype(str) == shock_id
    ]

    for other_id in other_shock_ids:
        forbidden_cols = [
            col for col in group.columns
            if col.endswith(f"_pre_{other_id}")
        ]

        for col in forbidden_cols:
            non_missing = int(group[col].notna().sum())

            if non_missing > 0:
                leakage_rows.append({
                    "shock_id": shock_id,
                    "forbidden_column": col,
                    "non_missing_rows": non_missing,
                })

snapshot_volatility_leakage_check = pd.DataFrame(leakage_rows)

snapshot_volatility_leakage_check.to_csv(
    DIAGNOSTICS_DIR / "snapshot_volatility_leakage_check.csv",
    index=False,
)

if not snapshot_volatility_leakage_check.empty:
    display(snapshot_volatility_leakage_check)
    raise ValueError("Cross-window volatility leakage detected in structural snapshot.")

# Put identifiers first.
front_cols = [
    "Country",
    "country_name",
    "shock_id",
    "analysis_year",
    "pre_shock_window",
    "Recovery",
    "recovery_outcome_column",
]

front_cols = [col for col in front_cols if col in master_structural_snapshot_by_shock.columns]
other_cols = [col for col in master_structural_snapshot_by_shock.columns if col not in front_cols]

master_structural_snapshot_by_shock = master_structural_snapshot_by_shock[
    front_cols + other_cols
].sort_values(["shock_id", "Country"]).reset_index(drop=True)

master_structural_snapshot_by_shock.to_csv(
    PROCESSED_DIR / "master_structural_snapshot_by_shock.csv",
    index=False,
)

print("Master structural snapshot rows:", len(master_structural_snapshot_by_shock))
print("Countries:", master_structural_snapshot_by_shock["Country"].nunique())
print("Shock IDs:", sorted(master_structural_snapshot_by_shock["shock_id"].astype(str).unique()))
print("Cross-window volatility leakage rows:", len(snapshot_volatility_leakage_check))

display(master_structural_snapshot_by_shock.head())

Master structural snapshot rows: 300
Countries: 162
Shock IDs: ['2007', '2019']
Cross-window volatility leakage rows: 0


,Country,country_name,shock_id,analysis_year,pre_shock_window,Recovery,recovery_outcome_column,Year,rd_gdp,inflation_cpi,unemployment_rate,tertiary_education_25_64,productivity_gdp_per_hour,gdp_growth,energy_import_dependency_raw,public_debt_gdp_canonical,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score,wgi_va_estimate,wgi_pv_estimate,wgi_ge_estimate,wgi_rq_estimate,wgi_rl_estimate,wgi_cc_estimate,wgi_complete_six_selected_dimensions,wgi_composite_input_type,wgi_euclidean_distance_to_ideal_selected_space,wgi_euclidean_ideal_score,wgi_mahalanobis_distance_to_ideal_full_wgi,wgi_mahalanobis_ideal_score_full_wgi,Recovery_2007,Recovery_2019,Recovery_2007_recovery_status,Recovery_2007_shock_detection_status,Recovery_2007_first_negative_growth_year,Recovery_2007_baseline_year,Recovery_2007_baseline_available_in_growth_file,Recovery_2007_recovery_year,Recovery_2007_years_from_baseline_to_recovery,Recovery_2007_years_after_first_negative_to_recovery,Recovery_2007_last_index_year,Recovery_2007_last_index_value,Recovery_2019_recovery_status,Recovery_2019_shock_detection_status,Recovery_2019_first_negative_growth_year,Recovery_2019_baseline_year,Recovery_2019_baseline_available_in_growth_file,Recovery_2019_recovery_year,Recovery_2019_years_from_baseline_to_recovery,Recovery_2019_years_after_first_negative_to_recovery,Recovery_2019_last_index_year,Recovery_2019_last_index_value,country_name_core,country_name_wgi,country_name_recovery,shock_label,window_start,window_end,gdp_growth_volatility_std_pre_2007,inflation_volatility_std_pre_2007,productivity_level_volatility_std_pre_2007,unemployment_volatility_std_pre_2007,gdp_growth_volatility_mad_pre_2007,inflation_volatility_mad_pre_2007,productivity_level_volatility_mad_pre_2007,unemployment_volatility_mad_pre_2007,gdp_growth_stability_negative_std_pre_2007,inflation_stability_negative_std_pre_2007,productivity_level_stability_negative_std_pre_2007,unemployment_stability_negative_std_pre_2007,gdp_growth_stability_negative_mad_pre_2007,inflation_stability_negative_mad_pre_2007,productivity_level_stability_negative_mad_pre_2007,unemployment_stability_negative_mad_pre_2007,gdp_growth_stability_std_0_100_pre_2007,inflation_stability_std_0_100_pre_2007,productivity_level_stability_std_0_100_pre_2007,unemployment_stability_std_0_100_pre_2007,gdp_growth_stability_mad_0_100_pre_2007,inflation_stability_mad_0_100_pre_2007,productivity_level_stability_mad_0_100_pre_2007,unemployment_stability_mad_0_100_pre_2007,gdp_growth_mean_value_pre_2007,inflation_mean_value_pre_2007,productivity_level_mean_value_pre_2007,unemployment_mean_value_pre_2007,gdp_growth_median_value_pre_2007,inflation_median_value_pre_2007,productivity_level_median_value_pre_2007,unemployment_median_value_pre_2007,gdp_growth_change_first_to_last_pre_2007,inflation_change_first_to_last_pre_2007,productivity_level_change_first_to_last_pre_2007,unemployment_change_first_to_last_pre_2007,gdp_growth_years_available_pre_2007,inflation_years_available_pre_2007,productivity_level_years_available_pre_2007,unemployment_years_available_pre_2007,gdp_growth_coverage_rate_in_window_pre_2007,inflation_coverage_rate_in_window_pre_2007,productivity_level_coverage_rate_in_window_pre_2007,unemployment_coverage_rate_in_window_pre_2007,gdp_growth_volatility_std_pre_2019,inflation_volatility_std_pre_2019,productivity_level_volatility_std_pre_2019,unemployment_volatility_std_pre_2019,gdp_growth_volatility_mad_pre_2019,inflation_volatility_mad_pre_2019,productivity_level_volatility_mad_pre_2019,unemployment_volatility_mad_pre_2019,gdp_growth_stability_negative_std_pre_2019,inflation_stability_negative_std_pre_2019,productivity_level_stability_negative_std_pre_2019,unemployment_stability_negative_std_pre_2019,gdp_growth_stability_negative_mad_pre_2019,inflation_stability_negative_mad_pre_2019,productivity_level_stability_negative_mad_pre_2019,unemployment_stability_negative_mad_pre_2019,gdp_growth_stability_std_0_100_pre_2019,inflati

In [12]:
# ------------------------------------------------------
# STRUCTURAL SNAPSHOT COVERAGE
# ------------------------------------------------------

snapshot_volatility_cols = [
    col for col in master_structural_snapshot_by_shock.columns
    if (
        col.startswith("gdp_growth_stability")
        or col.startswith("gdp_growth_volatility")
        or col.startswith("inflation_stability")
        or col.startswith("inflation_volatility")
        or col.startswith("unemployment_stability")
        or col.startswith("unemployment_volatility")
        or col.startswith("productivity_level_stability")
        or col.startswith("productivity_level_volatility")
    )
]

snapshot_candidate_columns = (
    MASTER_CORE_VARIABLES
    + wgi_composite_cols
    + snapshot_volatility_cols
    + ["Recovery"]
)

snapshot_candidate_columns = [
    col for col in snapshot_candidate_columns
    if col in master_structural_snapshot_by_shock.columns
]

snapshot_coverage_rows = []

for shock_id, group in master_structural_snapshot_by_shock.groupby("shock_id"):
    shock_id_str = str(shock_id)

    for col in snapshot_candidate_columns:
        # Only evaluate shock-specific volatility columns for their matching shock.
        # Example: pre_2007 features are only relevant to shock_id=2007.
        if "_pre_" in col and not col.endswith(f"_pre_{shock_id_str}"):
            continue

        values = group[col]

        snapshot_coverage_rows.append({
            "shock_id": shock_id,
            "column": col,
            "rows_non_missing": int(values.notna().sum()),
            "countries_total": group["Country"].nunique(),
            "coverage_rate": values.notna().mean(),
            "min_value": pd.to_numeric(values, errors="coerce").min(),
            "median_value": pd.to_numeric(values, errors="coerce").median(),
            "max_value": pd.to_numeric(values, errors="coerce").max(),
            "role": (
                "recovery_outcome_validation_only"
                if col == "Recovery"
                else "core_structural_source"
                if col in MASTER_CORE_VARIABLES
                else "wgi_composite"
                if col in wgi_composite_cols
                else "pre_shock_volatility_stability_candidate"
                if "_pre_" in col
                else "review"
            ),
        })

master_structural_snapshot_coverage = pd.DataFrame(snapshot_coverage_rows)

master_structural_snapshot_coverage.to_csv(
    DIAGNOSTICS_DIR / "master_structural_snapshot_coverage.csv",
    index=False,
)

display(master_structural_snapshot_coverage)

,shock_id,column,rows_non_missing,countries_total,coverage_rate,min_value,median_value,max_value,role
0,2007,rd_gdp,44,146,0.3014,0.1831,1.2675,4.2907,core_structural_source
1,2007,inflation_cpi,47,146,0.3219,0.0000,2.8527,10.0930,core_structural_source
2,2007,unemployment_rate,49,146,0.3356,2.4820,6.0750,34.9350,core_structural_source
3,2007,tertiary_education_25_64,37,146,0.2534,6.8958,28.9314,48.1816,core_structural_source
4,2007,productivity_gdp_per_hour,40,146,0.2740,14.0548,47.8268,118.7455,core_structural_source
5,2007,gdp_growth,44,146,0.3014,0.3324,5.1382,14.2308,core_structural_source
6,2007,energy_import_dependency_raw,130,146,0.8904,-1282.7102,24.7714,271.3164,core_structural_source
7,2007,public_debt_gdp_canonical,68,146,0.4658,3.9000,42.8893,117.5153,core_structural_source
8,2007,wgi_mahalanobis_ideal_score_full_wgi,144,146,0.9863,15.5352,52.6761,81.5177,wgi_composite
9,2007,wgi_euclidean_ideal_score,144,146,0.9863,25.0889,52.6915,90.2136,wgi_composite


In [13]:
# ------------------------------------------------------
# MASTER PROBLEM CASES
# ------------------------------------------------------

problem_rows = []

# Countries with structural snapshot but missing WGI composite.
if EXPECTED_WGI_COMPOSITE_COL in master_structural_snapshot_by_shock.columns:
    missing_wgi = master_structural_snapshot_by_shock[
        master_structural_snapshot_by_shock[EXPECTED_WGI_COMPOSITE_COL].isna()
    ][["Country", "country_name", "shock_id", "analysis_year"]].copy()

    for _, row in missing_wgi.iterrows():
        problem_rows.append({
            "problem_type": "missing_wgi_composite_in_shock_snapshot",
            "Country": row["Country"],
            "country_name": row["country_name"],
            "shock_id": row["shock_id"],
            "analysis_year": row["analysis_year"],
            "note": "WGI composite missing for country-year snapshot.",
        })

# Countries with structural snapshot but missing recovery.
missing_recovery = master_structural_snapshot_by_shock[
    master_structural_snapshot_by_shock["Recovery"].isna()
][["Country", "country_name", "shock_id", "analysis_year"]].copy()

for _, row in missing_recovery.iterrows():
    problem_rows.append({
        "problem_type": "missing_recovery_in_shock_snapshot",
        "Country": row["Country"],
        "country_name": row["country_name"],
        "shock_id": row["shock_id"],
        "analysis_year": row["analysis_year"],
        "note": "Recovery outcome missing; likely no GDP shock-window data or outside recovery country coverage.",
    })

# Countries with low core coverage in snapshot.
snapshot_core_count = master_structural_snapshot_by_shock[MASTER_CORE_VARIABLES].notna().sum(axis=1)
low_core = master_structural_snapshot_by_shock.loc[
    snapshot_core_count < 5,
    ["Country", "country_name", "shock_id", "analysis_year"]
].copy()
low_core["core_variables_available"] = snapshot_core_count[snapshot_core_count < 5].values

for _, row in low_core.iterrows():
    problem_rows.append({
        "problem_type": "low_core_structural_coverage_in_shock_snapshot",
        "Country": row["Country"],
        "country_name": row["country_name"],
        "shock_id": row["shock_id"],
        "analysis_year": row["analysis_year"],
        "note": f"Only {row['core_variables_available']} core variables available in structural snapshot.",
    })

master_problem_cases = pd.DataFrame(problem_rows)

if master_problem_cases.empty:
    master_problem_cases = pd.DataFrame(columns=[
        "problem_type",
        "Country",
        "country_name",
        "shock_id",
        "analysis_year",
        "note",
    ])

master_problem_cases.to_csv(
    DIAGNOSTICS_DIR / "master_problem_cases.csv",
    index=False,
)

display(master_problem_cases.head(100))

,problem_type,Country,country_name,shock_id,analysis_year,note
0,missing_wgi_composite_in_shock_snapshot,CUW,Curaçao,2007,2007,WGI composite missing for country-year snapshot.
1,missing_wgi_composite_in_shock_snapshot,SMR,San Marino,2007,2007,WGI composite missing for country-year snapshot.
2,missing_recovery_in_shock_snapshot,AGO,Angola,2007,2007,Recovery outcome missing; likely no GDP shock-...
3,missing_recovery_in_shock_snapshot,ALB,Albania,2007,2007,Recovery outcome missing; likely no GDP shock-...
4,missing_recovery_in_shock_snapshot,ARE,United Arab Emirates,2007,2007,Recovery outcome missing; likely no GDP shock-...
5,missing_recovery_in_shock_snapshot,ARG,Argentina,2007,2007,Recovery outcome missing; likely no GDP shock-...
6,missing_recovery_in_shock_snapshot,ARM,Armenia,2007,2007,Recovery outcome missing; likely no GDP shock-...
7,missing_recovery_in_shock_snapshot,AZE,Azerbaijan,2007,2007,Recovery outcome missing; likely no GDP shock-...
8,missing_recovery_in_shock_snapshot,BEN,Benin,2007,2007,Recovery outcome missing; likely no GDP shock-...
9,missing_recovery_in_shock_snapshot,BGD,Bangladesh,2007,2007,Recovery outcome missing; likely no GDP shock-...


In [14]:
# ------------------------------------------------------
# MASTER DATA DICTIONARY
# ------------------------------------------------------

dictionary_rows = []

for variable in MASTER_CORE_VARIABLES:
    dictionary_rows.append({
        "column": variable,
        "source": "Step 00 comparable long dataset",
        "role": "core_structural_source",
        "description": "Raw standardized structural variable. Direction alignment for POSet happens later.",
        "use_as_poset_ordering_variable_now": False,
        "validation_only": False,
    })

for col in wgi_score_cols:
    dictionary_rows.append({
        "column": col,
        "source": "WGI governance compilation",
        "role": "wgi_official_score_dimension",
        "description": "Official WGI 0-100 governance score dimension. Higher = better.",
        "use_as_poset_ordering_variable_now": False,
        "validation_only": False,
    })

for col in wgi_composite_cols:
    dictionary_rows.append({
        "column": col,
        "source": "WGI governance compilation",
        "role": "project_derived_governance_composite",
        "description": "Project-derived governance capacity composite. Not an official WGI indicator.",
        "use_as_poset_ordering_variable_now": False,
        "validation_only": False,
    })

for col in recovery_cols + ["Recovery"]:
    if col in master_country_year_panel.columns or col in master_structural_snapshot_by_shock.columns:
        dictionary_rows.append({
            "column": col,
            "source": "GDP recovery dynamic baseline notebook",
            "role": "recovery_outcome_validation_only",
            "description": "Observed recovery outcome. Must not be used as a POSet ordering variable.",
            "use_as_poset_ordering_variable_now": False,
            "validation_only": True,
        })

snapshot_volatility_cols_for_dictionary = [
    col for col in master_structural_snapshot_by_shock.columns
    if "_pre_" in col and ("stability" in col.lower() or "volatility" in col.lower())
]

for col in snapshot_volatility_cols_for_dictionary:
    dictionary_rows.append({
        "column": col,
        "source": "Volatility features notebook",
        "role": "pre_shock_volatility_or_stability_candidate",
        "description": "Shock-specific pre-shock volatility/stability feature. Candidate structural variable; review coverage and redundancy before POSet use.",
        "use_as_poset_ordering_variable_now": False,
        "validation_only": False,
    })

master_data_dictionary = pd.DataFrame(dictionary_rows).drop_duplicates(subset=["column"]).reset_index(drop=True)

master_data_dictionary.to_csv(
    PROCESSED_DIR / "master_country_year_panel_data_dictionary.csv",
    index=False,
)

master_data_dictionary.to_csv(
    DIAGNOSTICS_DIR / "master_country_year_panel_data_dictionary.csv",
    index=False,
)

methodological_notes = pd.DataFrame([
    {
        "topic": "Master panel base",
        "note": "The master country-year panel starts from standardized core comparable variables, then left-joins WGI, recovery outcomes, and volatility features.",
    },
    {
        "topic": "Public debt",
        "note": "Raw Eurostat and World Bank public debt source variables are excluded from the master core panel. Canonical public debt is used as the master public-debt variable.",
    },
    {
        "topic": "Recovery outcomes",
        "note": "Recovery_2007 and Recovery_2019 are merged for validation/interpretation only and must not be used as POSet ordering variables.",
    },
    {
        "topic": "WGI governance composite",
        "note": "The WGI Mahalanobis governance score is project-derived, not official WGI. The six WGI score dimensions remain available for audit/sensitivity.",
    },
    {
        "topic": "No final selection",
        "note": "This notebook does not choose the final country sample or final POSet variables. That happens in Pre-POSet EDA.",
    },
    {
        "topic": "No direction alignment",
        "note": "Most raw variables are not yet direction-aligned. Direction alignment is handled in the next notebook. Volatility features are kept only in the country-shock snapshot to avoid cross-window leakage.",
    },
])

methodological_notes.to_csv(
    DIAGNOSTICS_DIR / "master_methodological_notes.csv",
    index=False,
)

display(master_data_dictionary)
display(methodological_notes)

,column,source,role,description,use_as_poset_ordering_variable_now,validation_only
0,rd_gdp,Step 00 comparable long dataset,core_structural_source,Raw standardized structural variable. Directio...,False,False
1,inflation_cpi,Step 00 comparable long dataset,core_structural_source,Raw standardized structural variable. Directio...,False,False
2,unemployment_rate,Step 00 comparable long dataset,core_structural_source,Raw standardized structural variable. Directio...,False,False
3,tertiary_education_25_64,Step 00 comparable long dataset,core_structural_source,Raw standardized structural variable. Directio...,False,False
4,productivity_gdp_per_hour,Step 00 comparable long dataset,core_structural_source,Raw standardized structural variable. Directio...,False,False
5,gdp_growth,Step 00 comparable long dataset,core_structural_source,Raw standardized structural variable. Directio...,False,False
6,energy_import_dependency_raw,Step 00 comparable long dataset,core_structural_source,Raw standardized structural variable. Directio...,False,False
7,public_debt_gdp_canonical,Step 00 comparable long dataset,core_structural_source,Raw standardized structural variable. Directio...,False,False
8,wgi_va_score,WGI governance compilation,wgi_official_score_dimension,Official WGI 0-100 governance score dimension....,False,False
9,wgi_pv_score,WGI governance compilation,wgi_official_score_dimension,Official WGI 0-100 governance score dimension....,False,False


,topic,note
0,Master panel base,The master country-year panel starts from stan...
1,Public debt,Raw Eurostat and World Bank public debt source...
2,Recovery outcomes,Recovery_2007 and Recovery_2019 are merged for...
3,WGI governance composite,The WGI Mahalanobis governance score is projec...
4,No final selection,This notebook does not choose the final countr...
5,No direction alignment,Most raw variables are not yet direction-align...


In [15]:
# ------------------------------------------------------
# CREATE MASTER AUDIT WORKBOOK
# ------------------------------------------------------

MASTER_AUDIT_XLSX = AUDIT_DIR / "05_Master_Dataset_Build_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_merge_keys",
        "description": "Input source row/country/year diagnostics.",
        "path": DIAGNOSTICS_DIR / "master_merge_key_diagnostics.csv",
    },
    {
        "sheet_name": "02_country_overlap",
        "description": "Country overlap across core, WGI, recovery, and volatility sources.",
        "path": DIAGNOSTICS_DIR / "master_country_overlap_diagnostics.csv",
    },
    {
        "sheet_name": "03_country_gaps",
        "description": "Country merge gaps across sources.",
        "path": DIAGNOSTICS_DIR / "master_country_merge_gaps.csv",
    },
    {
        "sheet_name": "04_variable_coverage",
        "description": "Variable coverage in the master country-year panel.",
        "path": DIAGNOSTICS_DIR / "master_variable_coverage_summary.csv",
    },
    {
        "sheet_name": "05_country_coverage",
        "description": "Country coverage in the master country-year panel.",
        "path": DIAGNOSTICS_DIR / "master_country_coverage_summary.csv",
    },
    {
        "sheet_name": "06_snapshot_coverage",
        "description": "Coverage in the country-shock structural snapshot panel.",
        "path": DIAGNOSTICS_DIR / "master_structural_snapshot_coverage.csv",
    },
    {
        "sheet_name": "07_problem_cases",
        "description": "Potential merge/coverage problem cases.",
        "path": DIAGNOSTICS_DIR / "master_problem_cases.csv",
    },
    {
        "sheet_name": "08_dictionary",
        "description": "Master data dictionary.",
        "path": PROCESSED_DIR / "master_country_year_panel_data_dictionary.csv",
    },
    {
        "sheet_name": "09_method_notes",
        "description": "Methodological notes.",
        "path": DIAGNOSTICS_DIR / "master_methodological_notes.csv",
    },
]

used_sheets = set()

with pd.ExcelWriter(MASTER_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "05 Master Dataset Build Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for master country-year and country-shock dataset construction.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Master audit workbook created:")
print(MASTER_AUDIT_XLSX.resolve())

Master audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\05_Master_Dataset_Build_Audit.xlsx


In [16]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("05 MASTER DATASET BUILD COMPLETE")
print("=" * 80)

print("\nProcessed folder:")
print(PROCESSED_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nAudit workbook:")
print(MASTER_AUDIT_XLSX.resolve())

print("\nMain processed outputs:")
main_outputs = [
    "master_country_year_panel.csv",
    "master_country_year_panel_2000_2024.csv",
    "master_structural_snapshot_by_shock.csv",
    "master_country_year_panel_data_dictionary.csv",
]

for file_name in main_outputs:
    path = PROCESSED_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nMaster country-year panel:")
print("Rows:", len(master_country_year_panel))
print("Countries:", master_country_year_panel["Country"].nunique())
print("Years:", master_country_year_panel["Year"].min(), "-", master_country_year_panel["Year"].max())
print("Columns:", len(master_country_year_panel.columns))

print("\nMaster structural snapshot by shock:")
print("Rows:", len(master_structural_snapshot_by_shock))
print("Countries:", master_structural_snapshot_by_shock["Country"].nunique())
print("Shock IDs:", sorted(master_structural_snapshot_by_shock["shock_id"].astype(str).unique()))

print("\nImportant notes:")
print("1. No final country sample was chosen.")
print("2. No imputation or clipping was performed.")
print("3. Recovery outcomes were merged for validation only.")
print("4. Direction alignment and variable reduction happen next.")
print("5. Do not use Recovery as a POSet ordering variable.")

print("\nNext notebook:")
print("06_Pre_POSet_EDA_Checks.ipynb")

05 MASTER DATASET BUILD COMPLETE

Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Master_Dataset

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\05_Master_Dataset_Build

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\05_Master_Dataset_Build_Audit.xlsx

Main processed outputs:
- FOUND: master_country_year_panel.csv
- FOUND: master_country_year_panel_2000_2024.csv
- FOUND: master_structural_snapshot_by_shock.csv
- FOUND: master_country_year_panel_data_dictionary.csv

Master country-year panel:
Rows: 3584
Countries: 169
Years: 2000 - 2024
Columns: 54

Master structural snapshot by shock:
Rows: 300
Countries: 162
Shock IDs: ['2007', '2019']

Important notes:
1. No final country sample was chosen.
2. N